# Movie Recommendation System in Python

**Dataset:** [MovieLens 25M](https://grouplens.org/datasets/movielens/25m/) — 62,423 movies, 25 million ratings from 162,541 users  
**Guided by:** [Dataquest — Build a Movie Recommendation System in Python](https://app.dataquest.io/c/93/m/99994/build-a-movie-recommendation-system-in-python/1/introduction)  
**Techniques:** TF-IDF Vectorisation, Cosine Similarity, Collaborative Filtering  

---

This notebook implements a two-part system:
1. A **search engine** that finds movies by title using TF-IDF and cosine similarity
2. A **recommendation engine** that uses 25 million user ratings to surface films you are likely to enjoy based on the viewing habits of like-minded users

Both are connected to live Jupyter widgets that update as you type.

## Part 1: Content-Based Movie Search Engine

### 1.1 Load Movie Data

In [1]:
import pandas as pd

movies = pd.read_csv("ml-25m/movies.csv")
print(f"Loaded {len(movies):,} movies")
movies.head()

Loaded 62,423 movies


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


### 1.2 Clean Movie Titles Using Regex

Parentheses around release years — e.g., `(1995)` — and other non-alphanumeric characters create noise during tokenisation. We strip them with a single regex substitution, keeping only letters, digits, and spaces. The year is preserved as a plain token, which helps the vectoriser distinguish films with the same name released in different decades.

In [2]:
import re

def clean_title(title: str) -> str:
    """Remove all characters that are not letters, digits, or spaces."""
    return re.sub(r"[^a-zA-Z0-9 ]", "", title)

movies["clean_title"] = movies["title"].apply(clean_title)
movies.head()

,movieId,title,genres,clean_title
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story 1995
1,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji 1995
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men 1995
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Waiting to Exhale 1995
4,5,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II 1995


### 1.3 Build the TF-IDF Matrix

**TF-IDF** (Term Frequency — Inverse Document Frequency) converts each title string into a sparse numerical vector. Terms that appear in many titles (e.g., *the*, *a*) receive low weight; terms that are distinctive to a small number of titles receive high weight.

Setting `ngram_range=(1, 2)` captures both individual words and consecutive word pairs. This means *Toy Story 1995* yields unigrams `[toy, story, 1995]` and bigrams `[toy story, story 1995]`, giving the vectoriser more signal to separate closely named films.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(ngram_range=(1, 2))
tfidf = vectorizer.fit_transform(movies["clean_title"])

print(f"TF-IDF matrix shape: {tfidf.shape}")
print(f"Vocabulary size: {tfidf.shape[1]:,} unique n-grams")

TF-IDF matrix shape: (62423, 97591)
Vocabulary size: 97,591 unique n-grams


### 1.4 Cosine Similarity Search Function

At query time:
1. The user's input is cleaned and transformed into the same TF-IDF vector space
2. Cosine similarity is computed between the query vector and every row in the matrix
3. The five rows with highest similarity are returned

Using `np.argpartition` instead of `np.argsort` gives O(n) time for retrieving the top-k indices, avoiding a full sort of 62,423 scores.

In [4]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def search(title: str) -> pd.DataFrame:
    """
    Return the 5 movies whose cleaned titles are most similar
    to the query, ranked by cosine similarity.
    """
    cleaned = clean_title(title)
    query_vec = vectorizer.transform([cleaned])
    similarity = cosine_similarity(query_vec, tfidf).flatten()
    indices = np.argpartition(similarity, -5)[-5:]
    results = movies.iloc[indices].iloc[::-1]
    return results[["movieId", "title", "genres"]]

Quick test — confirm the function works before wiring it to the widget:

In [5]:
search("Toy Story")

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
3114,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy
78499,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX
168492,168492,Toy Story 4 (2019),Adventure|Animation|Children|Comedy|Fantasy
95510,95510,The Story of Film: An Odyssey (2011),Documentary


### 1.5 Interactive Movie Search Widget

The widget below updates the result table in real time as you type. No button press is required. The `observe` callback fires on every keystroke once the input length exceeds two characters.

In [6]:
import ipywidgets as widgets
from IPython.display import display

movie_input = widgets.Text(
    value="Toy Story",
    description="Search:",
    layout=widgets.Layout(width="420px"),
    style={"description_width": "80px"},
)
movie_list = widgets.Output()

def on_type(change):
    with movie_list:
        movie_list.clear_output(wait=True)
        title = change["new"]
        if len(title) > 2:
            display(search(title))

movie_input.observe(on_type, names="value")
display(movie_input, movie_list)

Text(value='Toy Story', description='Search:', layout=Layout(width='420px'), style=TextStyle(description_width='80px'))

Output()

---

## Part 2: Collaborative Filtering Recommendation Engine

### 2.1 Load Ratings Data

The ratings file contains 25,000,095 rows — one per user-movie interaction. Each record includes a `userId`, a `movieId`, a `rating` on a 0.5 to 5.0 scale (in 0.5 increments), and a Unix `timestamp`. Loading this file takes a few seconds.

In [7]:
ratings = pd.read_csv("ml-25m/ratings.csv")
print(f"Loaded {len(ratings):,} ratings from {ratings['userId'].nunique():,} unique users")
ratings.head()

Loaded 25,000,095 ratings from 162,541 unique users


,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


### 2.2 Prototype the Recommendation Logic

We prototype the algorithm step by step using *The Avengers (2012)* (movieId 89745) as a concrete example before generalising into a function.

**Step 1:** Find users who rated The Avengers above 4 stars.

In [8]:
movie_id = 89745  # The Avengers (2012)

similar_users = ratings[
    (ratings["movieId"] == movie_id) & (ratings["rating"] > 4)
]["userId"].unique()

print(f"Users who rated The Avengers (2012) above 4 stars: {len(similar_users):,}")

Users who rated The Avengers (2012) above 4 stars: 40,310


**Step 2:** Find other movies those users also loved, and compute what percentage of the cohort liked each one. Apply a 10% threshold to remove noise.

In [9]:
similar_user_recs = ratings[
    (ratings["userId"].isin(similar_users)) & (ratings["rating"] > 4)
]["movieId"]

similar_user_recs = similar_user_recs.value_counts() / len(similar_users)
similar_user_recs = similar_user_recs[similar_user_recs > 0.10]

print(f"Candidate movies after 10% threshold: {len(similar_user_recs)}")

Candidate movies after 10% threshold: 193


**Step 3:** Compute the same percentage across the entire user base and combine into a recommendation score.

In [10]:
all_users = ratings[
    (ratings["movieId"].isin(similar_user_recs.index)) & (ratings["rating"] > 4)
]
all_user_recs = all_users["movieId"].value_counts() / len(all_users["userId"].unique())

rec_percentages = pd.concat([similar_user_recs, all_user_recs], axis=1)
rec_percentages.columns = ["similar", "all"]
rec_percentages.head()

,similar,all
1,0.236083,0.126250
32,0.103877,0.101516
47,0.203115,0.146232
50,0.211067,0.202959
110,0.182240,0.162835


**Step 4:** Compute the score and show the top 10.

In [11]:
rec_percentages["score"] = rec_percentages["similar"] / rec_percentages["all"]
rec_percentages = rec_percentages.sort_values("score", ascending=False)

rec_percentages.head(10).merge(movies, left_index=True, right_on="movieId")

,similar,all,score,movieId,title,genres
17067,1.000000,0.040459,24.716368,89745,"Avengers, The (2012)",Action|Adventure|Sci-Fi|IMAX
20513,0.103711,0.005289,19.610199,106072,Thor: The Dark World (2013),Action|Adventure|Fantasy|IMAX
25058,0.241054,0.012367,19.491770,122892,Avengers: Age of Ultron (2015),Action|Adventure|Sci-Fi
19678,0.216534,0.012119,17.867419,102125,Iron Man 3 (2013),Action|Sci-Fi|Thriller|IMAX
16725,0.215043,0.012052,17.843074,88140,Captain America: The First Avenger (2011),Action|Adventure|Sci-Fi|Thriller|War
16312,0.175447,0.010142,17.299824,86332,Thor (2011),Action|Adventure|Drama|Fantasy|IMAX
21348,0.287608,0.016737,17.183667,110102,Captain America: The Winter Soldier (2014),Action|Adventure|Sci-Fi|IMAX
25071,0.214049,0.012856,16.649399,122920,Captain America: Civil War (2016),Action|Sci-Fi|Thriller
25061,0.136017,0.008573,15.865628,122900,Ant-Man (2015),Action|Adventure|Sci-Fi
14628,0.242876,0.015517,15.651921,77561,Iron Man 2 (2010),Action|Adventure|Sci-Fi|Thriller|IMAX


### 2.3 Recommendation Function

The logic above is wrapped into a reusable function. It accepts a `movieId` and returns the top 10 recommendations with three columns: `score`, `title`, and `genres`.

In [12]:
def find_similar_movies(movie_id: int) -> pd.DataFrame:
    """
    Given a movieId, return the top-10 recommended movies using collaborative filtering.

    Algorithm:
    1. Identify users who gave the target movie a rating > 4.0 (similar user cohort).
    2. Among that cohort, find all other movies they also rated > 4.0.
    3. Compute the percentage of the cohort that liked each candidate movie.
    4. Keep candidates liked by more than 10% of the cohort (noise threshold).
    5. Compute the same percentage across the full user population.
    6. Score each candidate as: score = cohort_percentage / all_users_percentage.
       A high score signals that a movie is specifically popular with this cohort,
       not merely broadly popular across all users.
    7. Return the top 10 candidates by score.
    """
    similar_users = ratings[
        (ratings["movieId"] == movie_id) & (ratings["rating"] > 4)
    ]["userId"].unique()

    similar_user_recs = ratings[
        (ratings["userId"].isin(similar_users)) & (ratings["rating"] > 4)
    ]["movieId"]
    similar_user_recs = similar_user_recs.value_counts() / len(similar_users)
    similar_user_recs = similar_user_recs[similar_user_recs > 0.10]

    all_users = ratings[
        (ratings["movieId"].isin(similar_user_recs.index)) & (ratings["rating"] > 4)
    ]
    all_user_recs = all_users["movieId"].value_counts() / len(all_users["userId"].unique())

    rec_percentages = pd.concat([similar_user_recs, all_user_recs], axis=1)
    rec_percentages.columns = ["similar", "all"]
    rec_percentages["score"] = rec_percentages["similar"] / rec_percentages["all"]
    rec_percentages = rec_percentages.sort_values("score", ascending=False)

    return (
        rec_percentages.head(10)
        .merge(movies, left_index=True, right_on="movieId")[["score", "title", "genres"]]
        .reset_index(drop=True)
    )

# Verify: Avengers should return MCU titles
find_similar_movies(89745)

,score,title,genres
0,24.716368,"Avengers, The (2012)",Action|Adventure|Sci-Fi|IMAX
1,19.610199,Thor: The Dark World (2013),Action|Adventure|Fantasy|IMAX
2,19.491770,Avengers: Age of Ultron (2015),Action|Adventure|Sci-Fi
3,17.867419,Iron Man 3 (2013),Action|Sci-Fi|Thriller|IMAX
4,17.843074,Captain America: The First Avenger (2011),Action|Adventure|Sci-Fi|Thriller|War
5,17.299824,Thor (2011),Action|Adventure|Drama|Fantasy|IMAX
6,17.183667,Captain America: The Winter Soldier (2014),Action|Adventure|Sci-Fi|IMAX
7,16.649399,Captain America: Civil War (2016),Action|Sci-Fi|Thriller
8,15.865628,Ant-Man (2015),Action|Adventure|Sci-Fi
9,15.651921,Iron Man 2 (2010),Action|Adventure|Sci-Fi|Thriller|IMAX


### 2.4 Full Interactive Recommendation Widget

The widget below chains both systems together:
1. User types a movie title
2. The search function matches it to the nearest title in the corpus
3. The top match's `movieId` is passed to `find_similar_movies`
4. The top 10 recommendations are displayed immediately

In [13]:
import ipywidgets as widgets
from IPython.display import display

movie_name_input = widgets.Text(
    value="Toy Story",
    description="Movie:",
    layout=widgets.Layout(width="440px"),
    style={"description_width": "80px"},
)
recommendation_list = widgets.Output()

def on_type(change):
    with recommendation_list:
        recommendation_list.clear_output(wait=True)
        title = change["new"]
        if len(title) > 2:
            results = search(title)
            if not results.empty:
                movie_id = results.iloc[0]["movieId"]
                display(find_similar_movies(movie_id))

movie_name_input.observe(on_type, names="value")

print("Type a movie title below to get personalised recommendations:")
display(movie_name_input, recommendation_list)

Text(value='Toy Story', description='Movie:', layout=Layout(width='440px'), style=TextStyle(description_width='80px'))

Output()

---

## Summary

| Component | Technique | Key Parameter |
|-----------|-----------|---------------|
| Title cleaning | Regex substitution | `[^a-zA-Z0-9 ]` |
| Vectorisation | TF-IDF | `ngram_range=(1,2)` |
| Search | Cosine similarity | Top-5 by score |
| Recommendation | Collaborative filtering | Rating threshold > 4.0, cohort threshold > 10% |
| Final ranking | Score ratio | `similar% / all%`, top 10 |

**Data credit:**  
F. Maxwell Harper and Joseph A. Konstan. 2015. *The MovieLens Datasets: History and Context.* ACM Transactions on Interactive Intelligent Systems, 5(4): 19:1-19:19.  
[https://grouplens.org/datasets/movielens/25m/](https://grouplens.org/datasets/movielens/25m/)

**Walkthrough credit:**  
[Dataquest — Build a Movie Recommendation System in Python](https://app.dataquest.io/c/93/m/99994/build-a-movie-recommendation-system-in-python/1/introduction)